In [1]:
import os

# Ensure TensorFlow uses the pure‑Python protobuf implementation to avoid GetPrototype errors
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, Model
import pandas as pd
import numpy as np




2026-05-03 00:48:13.000893: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-03 00:48:13.028971: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777769293.054538      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777769293.062874      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-05-03 00:48:13.093272: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [2]:
def train_and_predict():
    # Paths
    base_dir = "../input/plant-pathology-2020-fgvc7/images/"
    train_path = "../input/plant-pathology-2020-fgvc7/train.csv"
    test_path = "../input/plant-pathology-2020-fgvc7/test.csv"

    # Load CSVs and append .jpg to image names
    train_df = pd.read_csv(train_path)
    train_df["image_id"] = train_df["image_id"] + ".jpg"
    test_df = pd.read_csv(test_path)
    test_df["image_id"] = test_df["image_id"] + ".jpg"

    # Labels
    label_cols = ["healthy", "multiple_diseases", "rust", "scab"]

    # Image generator
    img_size = (128, 128)
    batch_size = 32
    datagen = ImageDataGenerator(rescale=1.0 / 255, validation_split=0.1)

    train_generator = datagen.flow_from_dataframe(
        dataframe=train_df,
        directory=base_dir,
        x_col="image_id",
        y_col=label_cols,
        target_size=img_size,
        batch_size=batch_size,
        class_mode="raw",
        shuffle=True,
        subset="training",
        seed=42,
    )

    val_generator = datagen.flow_from_dataframe(
        dataframe=train_df,
        directory=base_dir,
        x_col="image_id",
        y_col=label_cols,
        target_size=img_size,
        batch_size=batch_size,
        class_mode="raw",
        shuffle=False,
        subset="validation",
        seed=42,
    )

    # Build model – MobileNetV2 base (frozen) + simple head
    base_model = MobileNetV2(
        input_shape=img_size + (3,), include_top=False, weights="imagenet"
    )
    base_model.trainable = False

    x = layers.GlobalAveragePooling2D()(base_model.output)
    output = layers.Dense(len(label_cols), activation="sigmoid")(x)
    model = Model(inputs=base_model.input, outputs=output)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(),
        loss="binary_crossentropy",
        metrics=[tf.keras.metrics.AUC(name="auc")],
    )

    # Train (few epochs to stay within time limit)
    model.fit(train_generator, validation_data=val_generator, epochs=5, verbose=2)

    # Prepare test generator (no labels)
    test_datagen = ImageDataGenerator(rescale=1.0 / 255)
    test_generator = test_datagen.flow_from_dataframe(
        dataframe=test_df,
        directory=base_dir,
        x_col="image_id",
        y_col=None,
        target_size=img_size,
        batch_size=batch_size,
        class_mode=None,
        shuffle=False,
    )

    # Predict
    preds = model.predict(test_generator, verbose=0)
    pred_df = pd.DataFrame(preds, columns=label_cols)
    result = pd.concat([test_df[["image_id"]], pred_df], axis=1)
    # Remove the added ".jpg" suffix for submission
    result["image_id"] = result["image_id"].str.replace(".jpg", "")
    return result




In [3]:
if __name__ == "__main__":
    submission = train_and_predict()
    # Ensure column order matches sample submission
    submission = submission[
        ["image_id", "healthy", "multiple_diseases", "rust", "scab"]
    ]
    submission.to_csv("./submission.csv", index=False)
    print("Submission written to ./submission.csv")

Found 1475 validated image filenames.
Found 163 validated image filenames.


2026-05-03 00:48:22.892380: E external/local_xla/xla/stream_executor/cuda/cuda_driver.cc:152] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/5
47/47 - 28s - 601ms/step - auc: 0.8239 - loss: 0.4283 - val_auc: 0.8903 - val_loss: 0.3424
Epoch 2/5
47/47 - 20s - 424ms/step - auc: 0.9284 - loss: 0.2884 - val_auc: 0.9228 - val_loss: 0.2834
Epoch 3/5
47/47 - 19s - 411ms/step - auc: 0.9468 - loss: 0.2484 - val_auc: 0.9350 - val_loss: 0.2622
Epoch 4/5
47/47 - 21s - 436ms/step - auc: 0.9580 - loss: 0.2249 - val_auc: 0.9431 - val_loss: 0.2463
Epoch 5/5
47/47 - 20s - 433ms/step - auc: 0.9659 - loss: 0.2069 - val_auc: 0.9464 - val_loss: 0.2392
Found 183 validated image filenames.
Submission written to ./submission.csv
